In [0]:
# Databricks notebook source
# ══════════════════════════════════════
# BRONZE — physical_vendas_caixa
# Squad 3 — Arquitetura Medalhao
# Regra: copia fiel do dado bruto
#        sem alteracoes, sem tratamentos
# Particao: ano e mes (extraido de dt_venda)
# Frequencia: toda segunda-feira as 5h
# ══════════════════════════════════════

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# constantes do notebook

SOURCE_FILE  = "physical_vendas_caixa.csv"
SOURCE_PATH  = f"{RAW_BATCH_PATH}{SOURCE_FILE}"
BRONZE_TABLE = "physical_vendas_caixa"
BRONZE_PATH  = f"{BRONZE_BASE_PATH}{BRONZE_TABLE}"

EXPECTED_COLUMNS = [
    "id_transacao",
    "id_loja",
    "id_caixa",
    "id_operador",
    "dt_venda",
    "valor_total_venda",
    "cpf_cliente",
    "tipo_pagamento",
]

KEY_COLUMNS           = ["id_transacao"]
PARTITION_DATE_COLUMN = "dt_venda"
BRONZE_WRITE_MODE     = "overwrite"

print("Constantes configuradas:")
print(f"   SOURCE_PATH           : {SOURCE_PATH}")
print(f"   BRONZE_PATH           : {BRONZE_PATH}")
print(f"   KEY_COLUMNS           : {KEY_COLUMNS}")
print(f"   PARTITION_DATE_COLUMN : {PARTITION_DATE_COLUMN}")

In [0]:
# COMMAND ----------

adls_options = get_adls_options()
print("Opcoes ADLS configuradas.")

In [0]:
# ler CSV da Raw

df_source = read_source_csv(
    spark        = spark,
    source_path  = SOURCE_PATH,
    adls_options = adls_options,
    csv_options  = {
        "header"      : "true",
        "inferSchema" : "false",
    }
)

df_source.printSchema()
display(df_source.limit(10))

In [0]:
# contar origem

total_source = df_source.count()
print(f"Total de registros lidos da Raw: {total_source:,}")

In [0]:
# validar colunas esperadas

resultado_colunas = validate_required_columns(
    df               = df_source,
    expected_columns = EXPECTED_COLUMNS
)
print(resultado_colunas["message"])

if resultado_colunas["unexpected_columns"]:
    print(f"Colunas extras: {resultado_colunas['unexpected_columns']}")

In [0]:
# validar chave primaria na origem

resultado_pk = validate_key_columns(
    df          = df_source,
    key_columns = KEY_COLUMNS
)
print(resultado_pk["message"])

In [0]:
# validar data de particionamento

validate_partition_date(df_source, PARTITION_DATE_COLUMN)

In [0]:
# criar DataFrame Bronze
# copia fiel — sem alteracoes nos dados
# adiciona auditoria e ano/mes para particao

df_bronze = (
    df_source
    .select(
        *EXPECTED_COLUMNS,
        col("_metadata.file_path").alias("bronze_source_file")
    )
    .withColumn("bronze_ingested_at", current_timestamp())
    .withColumn("ano", year(to_timestamp(col(PARTITION_DATE_COLUMN))))
    .withColumn("mes", month(to_timestamp(col(PARTITION_DATE_COLUMN))))
)

# converte todos os campos para string
df_bronze = cast_all_columns_to_string(df_bronze)

print("Schema Bronze:")
df_bronze.printSchema()

display(
    df_bronze
    .select(
        "id_transacao",
        "id_loja",
        "dt_venda",
        "ano",
        "mes",
        "bronze_ingested_at",
        "bronze_source_file"
    )
    .limit(10)
)

In [0]:
# validar particoes criadas

display(
    df_bronze
    .groupBy("ano", "mes")
    .count()
    .orderBy("ano", "mes")
)

In [0]:
# validar Bronze antes de gravar

validate_bronze_quality(df_bronze, has_partitions=True)

In [0]:
# gravar Bronze Delta no ADLS
# particionado por ano e mes

write_delta(
    df           = df_bronze,
    path         = BRONZE_PATH,
    mode         = BRONZE_WRITE_MODE,
    partition_by = ["ano", "mes"],
    adls_options = adls_options
)

In [0]:
# ler Bronze gravada para validacao

df_bronze_saved = read_delta(
    spark        = spark,
    path         = BRONZE_PATH,
    adls_options = adls_options
)

df_bronze_saved.printSchema()
display(df_bronze_saved.limit(10))

In [0]:
# validar origem x Bronze

compare_row_counts(
    source_df = df_source,
    target_df = df_bronze_saved,
    label     = "Raw x Bronze physical_vendas_caixa"
)

In [0]:
# validar qualidade final da Bronze gravada

validate_bronze_quality(df_bronze_saved, has_partitions=True)

In [0]:
# validar particoes gravadas

display(
    df_bronze_saved
    .groupBy("ano", "mes")
    .count()
    .orderBy("ano", "mes")
)

In [0]:
# resumo final

print("=" * 55)
print("BRONZE physical_vendas_caixa concluida com sucesso!")
print("=" * 55)
print(f"""
   Source Path  : {SOURCE_PATH}
   Bronze Path  : {BRONZE_PATH}
   Total Raw    : {total_source:,}
   Total Bronze : {df_bronze_saved.count():,}
   Particao     : ano e mes (de dt_venda)
   Status       : SUCESSO
""")